# PlantCare RAG — Notebook 02 (all on Kaggle)One notebook, start to finish, no local machine required.```11 PDFs -> layout-aware parse -> 7 canonical sections -> bullet-item chunks        -> BGE-base embeddings -> Milvus Lite (dense FLAT + sparse BM25, one .db file)        -> hybrid retrieval + RRF -> tier routing        -> hosted open-weight LLM (Groq / OpenRouter, OpenAI-compatible)        -> closed-book generation with [S#] citations -> AUDIT -> extractive fallback        -> kb artefacts + retriever.py```## What replaced what| Was | Now | Why ||---|---|---|| Milvus Standalone in Docker | **Milvus Lite** — one embedded `.db` file | No Docker, no WSL, no 8 GB RAM ceiling. Verified in a test container: Lite supports `SPARSE_FLOAT_VECTOR` with `SPARSE_INVERTED_INDEX` **and** scalar filtering on both vector fields, so the full hybrid design survives intact. || Ollama `llama3.1:8b` on CPU | **Hosted open-weight model** over an OpenAI-compatible API | No 4.9 GB download, no 20-60 s per report on CPU. Still an open-source model — Llama / Qwen / GPT-OSS, not a closed one. |Milvus Lite is Linux-only, which is fine here — Kaggle is Linux. It's also why thisparticular design wouldn't have worked on your Windows machine.## Before you run1. **Settings → Internet → On** (needed for the model download and the LLM API).2. Get a **free API key** — no credit card:   - [console.groq.com](https://console.groq.com) → `GROQ_API_KEY` *(recommended: fastest, widest model list)*   - or [openrouter.ai](https://openrouter.ai) → `OPENROUTER_API_KEY`3. **Add-ons → Secrets → Add secret**, name it exactly `GROQ_API_KEY`, paste the key,   and tick "Attach to notebook".Use the Secrets store, not a variable in a cell. Kaggle notebooks are shareable and apasted key travels with the copy.Everything except the generation step works without a key — the notebook falls back toextractive reports and tells you so.

## 0. Setup

In [ ]:
import sys, subprocessdef pipi(*pkgs):    subprocess.run([sys.executable, "-m", "pip", "-q", "install", *pkgs], check=False)try:    import pymilvus, milvus_lite            # noqaexcept ImportError:    pipi("-U", "pymilvus", "milvus-lite")try:    import pymupdf                          # noqaexcept ImportError:    pipi("pymupdf")try:    import sentence_transformers            # noqaexcept ImportError:    pipi("sentence-transformers")import os, re, io, json, csv, math, time, textwrapfrom pathlib import Pathfrom collections import Counter, defaultdictimport numpy as np, requests, pandas as pd, pymupdf, pymilvusfrom pymilvus import MilvusClient, DataTypeprint("pymilvus", pymilvus.__version__)import torchprint("torch", torch.__version__, "| GPU:", torch.cuda.is_available())

## 1. Configuration

In [ ]:
# ---------------------------------------------------------------- pathsSEARCH_ROOTS = [Path("/kaggle/input"), Path("RAG_Documents"), Path(".")]_found = []for root in SEARCH_ROOTS:    if root.exists():        _found = sorted({p.parent for p in root.rglob("*.pdf")})        if _found:            breakassert _found, f"no PDFs under any of {[str(r) for r in SEARCH_ROOTS]} - attach the dataset"PDF_DIR = _found[0]if len(_found) > 1:    print("several PDF folders found, using the first:")    for f in _found:        print("   ", f, f"({len(list(f.glob('*.pdf')))} pdfs)")WORK    = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")OUT_DIR = WORK / "kb_export"; OUT_DIR.mkdir(parents=True, exist_ok=True)MILVUS_DB = str(WORK / "plantcare_kb.db")          # Milvus Lite = this one fileCOLLECTION = "plantcare_kb"# ---------------------------------------------------------------- classes (no wheat)CLASSES = [    "grape_black_measles", "grape_black_rot", "grape_healthy", "grape_isariopsis_leaf_spot",    "tomato_bacterial_spot", "tomato_early_blight", "tomato_healthy", "tomato_late_blight",    "tomato_leaf_mold", "tomato_septoria_leaf_spot", "tomato_spider_mites",    "tomato_target_spot", "tomato_yellowleaf_curl_virus",]HEALTHY_CLASSES = {"grape_healthy", "tomato_healthy"}# ---------------------------------------------------------------- the 7-section templateCANONICAL_HEADINGS = {    "disease name & target crop":                    "identity",    "scientific name (pathogen)":                    "pathogen",    "disease overview":                              "overview",    "causes & transmission vectors":                 "transmission",    "symptoms (early & advanced stages)":            "symptom",    "prevention & cultural practices":               "prevention",    "treatment & solutions (chemical & biological)": "treatment",}SECTION_TYPES = ["identity", "overview", "pathogen", "transmission",                 "symptom", "prevention", "treatment"]# ---------------------------------------------------------------- parsing (measured)TITLE_MIN_SIZE, HEAD_MIN_SIZE, HEAD_MAX_SIZE, LABEL_MAX_SIZE = 20.0, 13.0, 20.0, 12.5RUNNING_LINE    = "disease reference document for retrieval-augmented generation"CONTEXT_BELOW   = 140HARD_SPLIT_OVER = 900# ---------------------------------------------------------------- embeddingsST_MODEL     = "BAAI/bge-base-en-v1.5"EMBED_DIM    = 768QUERY_PREFIX = "Represent this sentence for searching relevant passages: "# ---------------------------------------------------------------- retrievalTOP_K_DENSE, TOP_K_SPARSE, RRF_K, FINAL_K, BLOCK_K = 20, 20, 60, 5, 3BM25_K1, BM25_B = 1.5, 0.75# ---------------------------------------------------------------- LLM providerPROVIDER = "groq"                    # "groq" or "openrouter"PROVIDERS = {    "groq": {        "base_url": "https://api.groq.com/openai/v1",        "secret": "GROQ_API_KEY",        # Preference order. Groq's catalogue churns, so the notebook picks the first        # of these that the API actually reports as live rather than hardcoding one.        "prefer": ["llama-3.3-70b-versatile", "openai/gpt-oss-120b",                   "qwen3-32b", "llama-3.1-8b-instant"],    },    "openrouter": {        "base_url": "https://openrouter.ai/api/v1",        "secret": "OPENROUTER_API_KEY",        "prefer": ["meta-llama/llama-3.3-70b-instruct:free",                   "qwen/qwen-2.5-72b-instruct:free",                   "meta-llama/llama-3.1-8b-instruct:free"],    },}LLM_TEMP    = 0.1        # low: copy the sources, don't be creativeREQ_PAUSE   = 6.0        # seconds between calls - free tiers cap tokens/minute, not just                         # requests/minute, and the per-minute token cap bites firstMAX_RETRIES = 3print("PDF_DIR   :", PDF_DIR)print("Milvus DB :", MILVUS_DB)print("provider  :", PROVIDER)

## 2. API key and model discoveryTwo things worth doing at runtime rather than hardcoding:* **The key comes from Kaggle Secrets**, so it isn't baked into a shared notebook.* **The model is discovered, not assumed.** Free-tier catalogues change — models get  added and removed with no notice. We ask the API what it serves today and take the  first entry from our preference list that's actually live.

In [ ]:
CFG = PROVIDERS[PROVIDER]API_KEY = ""try:    from kaggle_secrets import UserSecretsClient    API_KEY = UserSecretsClient().get_secret(CFG["secret"])except Exception as e:    API_KEY = os.environ.get(CFG["secret"], "")    if not API_KEY:        print(f"no {CFG['secret']} in Kaggle Secrets or env ({type(e).__name__})")LLM_OK, LLM_MODEL = False, Nonedef discover_model():    r = requests.get(f"{CFG['base_url']}/models", timeout=30,                     headers={"Authorization": f"Bearer {API_KEY}"})    r.raise_for_status()    live = [m["id"] for m in r.json().get("data", [])]    for want in CFG["prefer"]:        if want in live:            return want, live    chat = [m for m in live            if not re.search(r"whisper|guard|tts|embed|rerank|vision", m, re.I)]    return (chat[0] if chat else None), liveif API_KEY:    try:        LLM_MODEL, live = discover_model()        LLM_OK = LLM_MODEL is not None        print(f"{PROVIDER}: {len(live)} models live")        print("selected  :", LLM_MODEL)        missing = [m for m in CFG["prefer"] if m not in live]        if missing:            print("preferred but not currently served:", missing)    except Exception as e:        print(f"{PROVIDER} API check failed: {type(e).__name__}: {str(e)[:200]}")else:    print(f"No API key. Add-ons -> Secrets -> add '{CFG['secret']}' and attach it.")if not LLM_OK:    print("\n>> Reports will be EXTRACTIVE (verbatim cited passages). "          "Correct and citable, just not written in prose.")

## 3. Read the PDFsThe assert is the point of this cell. Every document in this corpus has a real textlayer, so there is no OCR path — if a scanned PDF ever lands in the folder we want a loudfailure here, not a silently empty section three cells later.

In [ ]:
pdf_paths = sorted(PDF_DIR.glob("*.pdf"))assert pdf_paths, f"no PDFs directly in {PDF_DIR}"print(f"{'file':46s} {'pp':>3s} {'chars':>7s}  text layer")print("-" * 78)no_text = []for p in pdf_paths:    d = pymupdf.open(p)    chars = sum(len(pg.get_text("text").strip()) for pg in d)    print(f"{p.name[:46]:46s} {len(d):3d} {chars:7d}  {'YES' if chars > 200 else 'NO -> SCANNED'}")    if chars <= 200:        no_text.append(p.name)    d.close()assert not no_text, f"no text layer in {no_text}. This notebook has no OCR path."print(f"\n{len(pdf_paths)} documents, all readable directly.")

## 4. Layout-aware parseMeasured on these exact files: the title is 24pt bold, the seven section headings are14pt bold with byte-identical wording across all 11 documents, and bullet labels are 11ptbold ending in a colon.The bullet detail matters. PyMuPDF dumps the `•` glyphs in a **detached block at the endof each page**, orphaned from their text — that's the long-standing "bullet structuredestroyed" bug. The glyphs are dropped; the real item boundary is the bold label.Wrapped lines are joined **keeping any trailing hyphen**, because all three line-endhyphens in this corpus (`post-bloom`, `broad-spectrum`, `store-bought`) are genuine, notline-wrap artefacts.

In [ ]:
def _norm_heading(s):    return re.sub(r"\s+", " ", s.lower().replace("&amp;", "&")).strip(" :")def read_lines(path):    doc = pymupdf.open(path)    out = []    for pno, page in enumerate(doc, start=1):        for block in page.get_text("dict")["blocks"]:            for line in block.get("lines", []):                spans = line["spans"]                text = "".join(s["text"] for s in spans).rstrip()                if not text.strip():                    continue                out.append({                    "text": text,                    "size": round(spans[0]["size"], 1),                    "bold": "bold" in spans[0]["font"].lower(),                    "bold_prefix": "".join(s["text"] for s in spans                                           if "bold" in s["font"].lower()).strip(),                    "page": pno,                })    doc.close()    return outdef is_noise(line):    t = line["text"].strip()    if not t or set(t) <= {"\u2022", " ", "\xa0"}:        return True    if RUNNING_LINE in t.lower():        return True    return bool(re.fullmatch(r"(page\s*)?\d+\s*(/|of)?\s*\d*", t, re.I))def join_wrapped(lines):    buf = ""    for ln in lines:        if buf.endswith("-") and re.match(r"[a-z]", ln):            buf += ln                       # keep the hyphen - real in this corpus        elif buf:            buf += " " + ln        else:            buf = ln    return re.sub(r"\s+", " ", buf).strip()def parse_document(path):    lines = [l for l in read_lines(path) if not is_noise(l)]    title, sections, cur = None, [], None    for ln in lines:        t = ln["text"].strip()        if title is None and ln["bold"] and ln["size"] >= TITLE_MIN_SIZE:            title = t; continue        if ln["bold"] and HEAD_MIN_SIZE <= ln["size"] < HEAD_MAX_SIZE:            stype = CANONICAL_HEADINGS.get(_norm_heading(t))            if stype:                cur = {"heading": t, "section_type": stype, "items": [], "page": ln["page"]}                sections.append(cur); continue        if cur is None:            continue        bp = ln["bold_prefix"]        new_item = (bp and ":" in bp and ln["size"] <= LABEL_MAX_SIZE                    and t.lower().startswith(bp.lower()[:8]))        if new_item or not cur["items"]:            cur["items"].append([t])        else:            cur["items"][-1].append(t)    for sec in sections:        sec["items"] = [x for x in (join_wrapped(i) for i in sec["items"]) if x]    return title, sectionsparsed = {}print(f"{'file':46s} {'title':30s} secs items")print("-" * 92)for p in pdf_paths:    title, secs = parse_document(p)    parsed[p.name] = {"title": title, "sections": secs}    print(f"{p.name[:46]:46s} {str(title)[:30]:30s} {len(secs):4d} "          f"{sum(len(s['items']) for s in secs):5d}")    gap = [v for v in SECTION_TYPES if v not in {s["section_type"] for s in secs}]    if gap:        print(f"    !! missing sections: {gap}")

### Eyeball the parse before trusting itEvery item should be a complete, self-contained passage starting with its bold label. Afragment beginning mid-word means the item-boundary rule needs work.

In [ ]:
name = pdf_paths[0].nameprint(f"=== {name} - {parsed[name]['title']} ===")for sec in parsed[name]["sections"]:    print(f"\n### {sec['section_type']}   ({sec['heading']})")    for item in sec["items"]:        print(f"  [{len(item):4d}] {item}")

## 5. Class taggingFrom the filename, exactly. Comparison strips every non-alphanumeric character — which iswhat makes `Tomato_Yellow_Leaf_Curl_Virus_RAG.pdf` match the class`tomato_yellowleaf_curl_virus` despite the different underscore placement.

In [ ]:
def _key(s):    return re.sub(r"[^a-z0-9]", "", s.lower())CLASS_LOOKUP = {_key(c): c for c in CLASSES}def classify_document(filename):    stem = re.sub(r"_?rag(_document)?$", "", Path(filename).stem, flags=re.I)    return CLASS_LOOKUP.get(_key(stem))doc_class, unmatched = {}, []for n in parsed:    cls = classify_document(n)    doc_class[n] = cls    print(f"{n[:48]:48s} -> {cls}")    if cls is None:        unmatched.append(n)assert not unmatched, f"could not map to a class: {unmatched}"

## 6. Chunking — one bullet, one chunkMeasured: 172 items, median 155 chars, max 542. Each item is already a coherent passagewritten by whoever authored these documents; splitting or merging them would only makeretrieval worse.**Display text and retrieval text are separate fields**, which matters more than it sounds:* `text` (what a farmer reads) gets only a `"<Disease>: "` prefix when the item is short.  The report already prints the section label above it, so repeating the heading gave  `"Tomato Early Blight - Symptoms: Foliar Symptoms: ..."` — three levels of nesting.* `search_text` (what the index sees) keeps the full heading, which is real dense-retrieval  signal, plus extracted pathogen binomials and chemical names so a BM25 query for  *Xanthomonas* hits a chunk that only says "the bacteria".

In [ ]:
BINOMIAL_RE = re.compile(r"\b([A-Z][a-z]{3,})\s+([a-z]{3,})\b")CHEM_RE = re.compile(    r"\b(mancozeb|chlorothalonil|copper|captan|myclobutanil|azoxystrobin|difenoconazole|"    r"tebuconazole|propiconazole|streptomycin|bordeaux|sulfur|sulphur|abamectin|spinosad|"    r"imidacloprid|spiromesifen|neem|bacillus|trichoderma|maneb|ziram|fosetyl|mefenoxam|"    r"metalaxyl|acibenzolar|actigard|phytoseiulus)\b", re.I)def split_sentences(t):    return [s.strip() for s in re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", t) if s.strip()]def hard_split(item):    if len(item) <= HARD_SPLIT_OVER:        return [item]    out, buf = [], ""    for s in split_sentences(item):        if buf and len(buf) + len(s) > HARD_SPLIT_OVER:            out.append(buf); buf = s        else:            buf = f"{buf} {s}".strip()    if buf:        out.append(buf)    return outchunks = []for name, info in parsed.items():    cls  = doc_class[name]    crop = cls.split("_")[0]    pretty_name = (info["title"] or cls.replace("_", " ")).title()    for sec in info["sections"]:        for item in sec["items"]:            for piece in hard_split(item):                rescued = len(piece) < CONTEXT_BELOW                body = f"{pretty_name}: {piece}" if rescued else piece                retr = f"{pretty_name} - {sec['heading']}: {piece}"                kws  = {f"{m.group(1)} {m.group(2)}" for m in BINOMIAL_RE.finditer(piece)}                kws |= {m.group(0).lower() for m in CHEM_RE.finditer(piece)}                chunks.append({                    "id": len(chunks), "text": body,                    "search_text": retr + (" " + " ".join(sorted(kws)) if kws else ""),                    "keywords": sorted(kws), "disease": cls, "crop": crop,                    "section_type": sec["section_type"], "section_title": sec["heading"],                    "rescued": rescued, "doc": name, "page": sec["page"],                })lens = sorted(len(c["text"]) for c in chunks)print(f"{len(chunks)} chunks | min {lens[0]} | median {lens[len(lens)//2]} | max {lens[-1]}")print("by section:", dict(Counter(c["section_type"] for c in chunks)))print("by crop   :", dict(Counter(c["crop"] for c in chunks)))

## 7. Coverage and validation gate

In [ ]:
REQUIRED = ["symptom", "pathogen", "transmission", "prevention", "treatment"]by_class = defaultdict(list)for c in chunks:    by_class[c["disease"]].append(c)coverage = {}print(f"{'class':32s} {'chunks':>7s} {'verdict':9s}  missing")print("-" * 80)for cls in CLASSES:    cs = by_class.get(cls, [])    have = {c["section_type"] for c in cs}    missing = [s for s in REQUIRED if s not in have]    verdict = ("HEALTHY" if cls in HEALTHY_CLASSES else               "MISSING" if not cs else               "THIN" if missing or len(cs) < 6 else "OK")    coverage[cls] = {"chunks": len(cs), "verdict": verdict,                     "sections": sorted(have), "missing": missing}    print(f"{cls:32s} {len(cs):7d} {verdict:9s}  {', '.join(missing) or '-'}")print("\n", dict(Counter(v["verdict"] for v in coverage.values())))problems = []if not chunks:    problems.append("no chunks produced")if any(not c["disease"] for c in chunks):    problems.append("untagged chunks")for cls, v in coverage.items():    if cls in HEALTHY_CLASSES:        continue    if v["verdict"] == "MISSING":        problems.append(f"{cls}: no source document")    elif v["missing"]:        problems.append(f"{cls}: missing {v['missing']}")frag = [c for c in chunks if re.match(r"^[a-z]", c["text"]) and not c["rescued"]]if frag:    problems.append(f"{len(frag)} chunks start mid-sentence, e.g. {frag[0]['text'][:60]!r}")if problems:    print("\nGATE FAILED:")    for p in problems:        print("  -", p)    raise AssertionError("fix the above before continuing")print(f"\nGATE PASSED - {len(chunks)} chunks, "      f"{sum(1 for v in coverage.values() if v['verdict'] == 'OK')} classes fully covered")

## 8. Dense embeddings`bge-base-en-v1.5` is **asymmetric**: the query gets an instruction prefix, the passagedoesn't. That fits this workload — the query is a short machine-built string from theCNN's class label, the passage is a full paragraph. A symmetric model treats both as thesame kind of text and loses accuracy on exactly that mismatch.

In [ ]:
_ST = Nonedef embed(texts, is_query=False, batch=32):    global _ST    if _ST is None:        from sentence_transformers import SentenceTransformer        dev = "cuda" if torch.cuda.is_available() else "cpu"        print(f"loading {ST_MODEL} on {dev} (first run downloads ~440 MB) ...")        _ST = SentenceTransformer(ST_MODEL, device=dev)    if is_query:        texts = [QUERY_PREFIX + t for t in texts]    v = _ST.encode(texts, batch_size=batch, normalize_embeddings=True,                   show_progress_bar=len(texts) > 64)    return np.asarray(v, dtype=np.float32)dense = embed([c["text"] for c in chunks])assert dense.shape == (len(chunks), EMBED_DIM), dense.shapeassert np.allclose(np.linalg.norm(dense, axis=1), 1.0, atol=1e-4)print("dense:", dense.shape, "unit-norm OK")

## 9. BM25 as a sparse vectorMilvus stores a sparse vector as `{term_id: weight}` and scores it with inner product. Wemake that inner product *be* the BM25 score by splitting the formula:* **document side** stores `tf·(k1+1) / (tf + k1·(1 − b + b·dl/avgdl))`* **query side** stores `IDF(term)`Their dot product is the standard BM25 sum. The cell proves it against a direct BM25implementation rather than asking you to trust the algebra.

In [ ]:
TOKEN_RE = re.compile(r"[a-z0-9]+")def tokenize(s):    return TOKEN_RE.findall(s.lower())docs_tok = [tokenize(c["search_text"]) for c in chunks]N, avgdl = len(docs_tok), sum(len(d) for d in docs_tok) / len(docs_tok)df = Counter()for d in docs_tok:    df.update(set(d))vocab = {t: i for i, t in enumerate(sorted(df))}idf   = {t: math.log(1 + (N - df[t] + 0.5) / (df[t] + 0.5)) for t in df}def doc_sparse(tokens):    tf, dl = Counter(tokens), len(tokens)    return {vocab[t]: float(f * (BM25_K1 + 1) /                            (f + BM25_K1 * (1 - BM25_B + BM25_B * dl / avgdl)))            for t, f in tf.items()}def query_sparse(text):    return {vocab[t]: float(idf[t]) for t in set(tokenize(text)) if t in vocab}sparse_docs = [doc_sparse(d) for d in docs_tok]def bm25_direct(qtok, dtok):    tf, dl = Counter(dtok), len(dtok)    return sum(idf[t] * tf[t] * (BM25_K1 + 1) /               (tf[t] + BM25_K1 * (1 - BM25_B + BM25_B * dl / avgdl))               for t in set(qtok) if t in df and t in tf)q  = "early blight concentric rings bullseye lower leaves"qs, qt = query_sparse(q), tokenize(q)worst = max(abs(sum(w * sparse_docs[i].get(t, 0.0) for t, w in qs.items())                - bm25_direct(qt, d)) for i, d in enumerate(docs_tok))print(f"vocab {len(vocab)} | avgdl {avgdl:.1f} | max |IP - BM25| = {worst:.2e}")assert worst < 1e-5, "sparse encoding does not reproduce BM25"print("sparse encoding verified")

## 10. Milvus Lite`MilvusClient(uri="something.db")` runs Milvus **in this process**, backed by a singlefile. No server, no Docker, no ports.The collection holds both halves: a 768-d dense vector on `FLAT`/IP, and a sparse vectoron `SPARSE_INVERTED_INDEX`/IP. `FLAT` is exact brute force — with 172 chunks that is theright call, since `IVF_FLAT` and `HNSW` are approximations buying speed we don't need ata recall cost we can't spare.The sparse field is created inside a `try`. It works on current Milvus Lite (verified),but Lite has historically lagged the server on vector types, so if your version rejectsit the notebook drops to a dense-only collection and runs BM25 in Python instead. Sameresults either way — it just prints which path it took.

In [ ]:
if os.path.exists(MILVUS_DB):    os.remove(MILVUS_DB)client = MilvusClient(uri=MILVUS_DB)def build_collection(with_sparse):    if client.has_collection(COLLECTION):        client.drop_collection(COLLECTION)    s = client.create_schema(auto_id=False, enable_dynamic_field=False)    s.add_field("id",            DataType.INT64, is_primary=True)    s.add_field("text",          DataType.VARCHAR, max_length=4000)    s.add_field("disease",       DataType.VARCHAR, max_length=64)    s.add_field("crop",          DataType.VARCHAR, max_length=32)    s.add_field("section_type",  DataType.VARCHAR, max_length=32)    s.add_field("section_title", DataType.VARCHAR, max_length=256)    s.add_field("doc",           DataType.VARCHAR, max_length=256)    s.add_field("page",          DataType.INT64)    s.add_field("rescued",       DataType.BOOL)    s.add_field("dense",         DataType.FLOAT_VECTOR, dim=EMBED_DIM)    if with_sparse:        s.add_field("sparse",    DataType.SPARSE_FLOAT_VECTOR)    ix = client.prepare_index_params()    ix.add_index(field_name="dense", index_type="FLAT", metric_type="IP")    if with_sparse:        ix.add_index(field_name="sparse", index_type="SPARSE_INVERTED_INDEX",                     metric_type="IP")    client.create_collection(COLLECTION, schema=s, index_params=ix)    rows = []    for i, c in enumerate(chunks):        row = {"id": int(c["id"]), "text": c["text"][:4000],               "disease": c["disease"], "crop": c["crop"],               "section_type": c["section_type"],               "section_title": (c.get("section_title") or "")[:256],               "doc": c["doc"][:256], "page": int(c["page"]),               "rescued": bool(c.get("rescued", False)),               "dense": dense[i].tolist()}        if with_sparse:            row["sparse"] = sparse_docs[i]        rows.append(row)    client.insert(COLLECTION, rows)    client.load_collection(COLLECTION)SPARSE_IN_MILVUS = Truetry:    build_collection(True)    client.search(COLLECTION, data=[query_sparse("early blight")], anns_field="sparse",                  limit=1, output_fields=["id"], search_params={"metric_type": "IP"})    print("Milvus Lite: dense + sparse, both in the collection")except Exception as e:    SPARSE_IN_MILVUS = False    print("sparse rejected by this Milvus Lite build:", type(e).__name__, str(e)[:150])    print("-> dense in Milvus, BM25 in Python")    build_collection(False)print("stats:", client.get_collection_stats(COLLECTION))print("db file:", MILVUS_DB, f"({os.path.getsize(MILVUS_DB)/1024:.0f} KB)")

## 11. Hybrid retrieval + Reciprocal Rank FusionRRF fuses **ranks**, not scores: `score = Σ 1/(k + rank)`. That's the point — a cosinesimilarity lives in [−1, 1] and a BM25 sum is unbounded, so there is no honest constantto normalise between them. Positions are comparable by construction.We fuse in Python rather than using Milvus's `RRFRanker` because the block logic needs adifferent filter expression per query and occasionally has to suppress a whole sectiontype.

In [ ]:
by_id = {c["id"]: c for c in chunks}def _match(c, expr):    if not expr:        return True    for clause in expr.split(" and "):        m = re.match(r'\s*(\w+)\s*(==|!=)\s*"(.*)"\s*$', clause)        assert m, f"unsupported filter clause: {clause}"        field, op, val = m.groups()        got = c.get(field) or ""        if (op == "==" and got != val) or (op == "!=" and got == val):            return False    return Truedef search_dense(qvec, expr, k):    r = client.search(COLLECTION, data=[qvec.tolist()], anns_field="dense", limit=k,                      filter=expr or "", output_fields=["id"],                      search_params={"metric_type": "IP"})    return [h["id"] for h in r[0]]def search_sparse(qsp, expr, k):    if not qsp:        return []    if SPARSE_IN_MILVUS:        r = client.search(COLLECTION, data=[qsp], anns_field="sparse", limit=k,                          filter=expr or "", output_fields=["id"],                          search_params={"metric_type": "IP"})        return [h["id"] for h in r[0]]    scored = []    for i, sd in enumerate(sparse_docs):        if not _match(chunks[i], expr):            continue        s = sum(w * sd.get(t, 0.0) for t, w in qsp.items())        if s > 0:            scored.append((s, chunks[i]["id"]))    scored.sort(key=lambda x: -x[0])    return [i for _, i in scored[:k]]def rrf(*rank_lists, k=RRF_K):    score = defaultdict(float)    for lst in rank_lists:        for rank, doc_id in enumerate(lst, start=1):            score[doc_id] += 1.0 / (k + rank)    return sorted(score.items(), key=lambda x: -x[1])def hybrid_search(query, expr=None, k=FINAL_K):    qv = embed([query], is_query=True)[0]    d = search_dense(qv, expr, TOP_K_DENSE)    s = search_sparse(query_sparse(query), expr, TOP_K_SPARSE)    return [dict(by_id[i], _rrf=sc) for i, sc in rrf(d, s)[:k]]for h in hybrid_search("concentric rings bullseye on older lower tomato leaves"):    print(f"[{h['_rrf']:.4f}] {h['disease']:30s} {h['section_type']:12s} {h['text'][:60]}...")

## 12. Report blocks and tier routingFive blocks, each backed by its own filtered query against one section type. Cross-blockdedup means a passage used by Symptoms can't reappear under Cause.| Tier | When | May return ||---|---|---|| **A** | the class has its own document | everything || **B** | no document — fall back to the same crop | **no treatment passages**, plus a caveat || **C** | nothing at all | no report |**Why B blocks treatment.** A class with no document would fall back to its nearest cropneighbour, which might be a bacterial disease when it is a fungal one — copper andstreptomycin versus a strobilurin. A tier-B answer would print the wrong chemistry underthe right heading, with a real citation attached. That is worse than no answer, becauseit *looks* verified. On this corpus no class is in tier B; the assert confirms it andkeeps the hole closed if a 14th class is ever added without a document.

In [ ]:
SECTION_QUERIES = {    "symptoms":   ("symptom",      "{d} symptoms on leaves early and advanced signs lesions"),    "cause":      ("pathogen",     "{d} causal pathogen scientific name fungus bacterium virus"),    "reason":     ("transmission", "{d} causes transmission how it spreads overwintering conditions"),    "precaution": ("prevention",   "{d} prevention cultural practices sanitation resistant varieties"),    "treatment":  ("treatment",    "{d} treatment chemical biological control fungicide solutions"),}BLOCK_LABELS = {"symptoms": "Symptoms", "cause": "Cause", "reason": "How it spreads",                "precaution": "Prevention", "treatment": "Treatment"}def pretty(cls):    return cls.replace("_", " ")def tier_for(cls):    if cls in HEALTHY_CLASSES:        return "healthy"    if coverage.get(cls, {}).get("verdict") in ("OK", "THIN"):        return "A"    crop = cls.split("_")[0]    if any(c != cls and c not in HEALTHY_CLASSES and c.startswith(crop + "_")           and coverage[c]["verdict"] in ("OK", "THIN") for c in coverage):        return "B"    return "C"def retrieve_blocks(cls, k=BLOCK_K):    tier = tier_for(cls)    if tier == "C":        return tier, {}, ["No source document covers this class."]    caveats = []    if tier == "A":        expr = f'disease == "{cls}"'    else:        crop = cls.split("_")[0]        expr = f'crop == "{crop}" and disease != "{cls}"'        caveats.append(            f"No document in the knowledge base covers {pretty(cls)}. The passages below "            f"describe other {crop} diseases and are general context only. Treatment "            f"guidance has been withheld because it would not be specific to this disease.")    blocks, seen = {}, set()    for name, (stype, tmpl) in SECTION_QUERIES.items():        if tier == "B" and stype == "treatment":            continue                                    # <-- the safety rule        query = tmpl.format(d=pretty(cls))        hits = hybrid_search(query, expr=f'{expr} and section_type == "{stype}"', k=k)        if not hits:            hits = hybrid_search(query, expr=expr, k=k)        keep = [h for h in hits if h["id"] not in seen]        seen.update(h["id"] for h in keep)        if keep:            blocks[name] = keep    return tier, blocks, caveatsprint(f"{'class':32s} {'tier':7s} blocks")print("-" * 88)empty = []for cls in CLASSES:    t, b, cav = retrieve_blocks(cls)    print(f"{cls:32s} {t:7s} {list(b)}")    if t == "B":        assert "treatment" not in b, f"SAFETY: tier B returned treatment for {cls}"        assert cav, f"SAFETY: tier B returned no caveat for {cls}"    if t == "A":        miss = [k for k in SECTION_QUERIES if k not in b]        if miss:            empty.append((cls, miss))assert not empty, f"tier-A classes with empty blocks: {empty}"print("\nall tier-A classes fill all five blocks; tier-B safety rule holds")

## 13. Generation over the hosted APIStandard OpenAI-compatible `/chat/completions`, so the same code works for Groq,OpenRouter, or anything else that speaks that shape.Free tiers cap **tokens per minute** as well as requests per minute, and the token capusually bites first — so calls are spaced by `REQ_PAUSE` and a `429` is retried withbackoff, honouring `Retry-After` when the server sends it.

In [ ]:
SYSTEM = (    "You are an agricultural extension assistant. You answer ONLY from the numbered "    "sources given to you. You never use outside knowledge, never estimate, and never "    "invent a chemical name, a dose, a rate or an interval. If the sources do not "    "contain something, write exactly: 'The sources do not state this.' "    "Every sentence must end with one or more citation markers like [S1] or [S2][S3].")def build_prompt(cls, blocks, caveats):    src, meta = [], []    for name, hits in blocks.items():        for h in hits:            meta.append(h)            src.append(f"[S{len(meta)}] ({name} | {h['doc']} p.{h['page']}) {h['text']}")    cav = ("\n\nIMPORTANT CONTEXT: " + " ".join(caveats)) if caveats else ""    headings = "\n".join(f"{l}:" for l in BLOCK_LABELS.values())    return (f"Disease identified by the image classifier: {pretty(cls)}\n\n"            f"SOURCES:\n" + "\n\n".join(src) + cav + "\n\n"            f"Write a short field advisory with exactly these headings:\n{headings}\n\n"            "One to three sentences per heading, in plain language a farmer can act on. "            "Cite after every sentence. If a heading has no supporting source, write "            "'The sources do not state this.'"), meta_last_call = [0.0]def ask_llm(user, system=SYSTEM):    for attempt in range(MAX_RETRIES):        wait = REQ_PAUSE - (time.time() - _last_call[0])        if wait > 0:            time.sleep(wait)        _last_call[0] = time.time()        r = requests.post(f"{CFG['base_url']}/chat/completions", timeout=120,            headers={"Authorization": f"Bearer {API_KEY}",                     "Content-Type": "application/json"},            json={"model": LLM_MODEL, "temperature": LLM_TEMP,                  "messages": [{"role": "system", "content": system},                               {"role": "user", "content": user}]})        if r.status_code == 429:            back = float(r.headers.get("Retry-After", REQ_PAUSE * (attempt + 2)))            print(f"    rate limited, waiting {back:.0f}s")            time.sleep(back)            continue        r.raise_for_status()        return r.json()["choices"][0]["message"]["content"]    raise RuntimeError("rate limited after retries")

## 14. The auditA model will happily write *"apply mancozeb at 2.5 g/litre [S2]"* when S2 says nothing ofthe sort — the citation marker is just text to it. So we verify mechanically: every**number carrying a unit** and every **chemical-looking token** must appear in the sourcetext, and every `[S#]` must exist.Anything unsupported fails the audit and we ship the extractive report instead. A clunkieranswer that is true beats a fluent one that is invented.

In [ ]:
NUM_UNIT_RE = re.compile(    r"\b\d+(?:[.,]\d+)?\s*(?:-\s*\d+(?:[.,]\d+)?\s*)?"    r"(?:%|g|kg|mg|ml|l|litre|liter|lb|oz|ha|acre|day|days|week|weeks|hour|hours|year|years|"    r"ppm|mesh|cm|mm|inch|inches|°c|°f)\b", re.I)def _norm(s):    return re.sub(r"[\s,]+", " ", s.lower()).strip()def audit_answer(answer, meta):    source_text = _norm(" ".join(h["text"] for h in meta))    compact = source_text.replace(" ", "")    problems = []    if not re.search(r"\[S\d+\]", answer):        problems.append("no citation markers at all")    for m in NUM_UNIT_RE.finditer(answer):        frag = _norm(m.group(0))        if frag not in source_text and frag.replace(" ", "") not in compact:            problems.append(f"unsupported quantity: {m.group(0).strip()!r}")    for m in CHEM_RE.finditer(answer):        if m.group(0).lower() not in source_text:            problems.append(f"unsupported chemical: {m.group(0)!r}")    for m in re.finditer(r"\[S(\d+)\]", answer):        if not (1 <= int(m.group(1)) <= len(meta)):            problems.append(f"citation [S{m.group(1)}] does not exist")    return (not problems), problems# ---- self-test against a real passage from your corpus ------------------_m = [{"text": "Chemical Treatments: Effective fungicides include protectants like "                "Mancozeb and Captan, and systemic fungicides like Myclobutanil and "                "Tebuconazole (sterol inhibitors).", "doc": "t.pdf", "page": 2}]ok, p = audit_answer("Use protectants such as Mancozeb or Captan [S1].", _m)assert ok, pprint("PASS grounded text")ok, p = audit_answer("Apply azoxystrobin at 2.5 g per litre every 7 days [S1].", _m)assert not ok and any("azoxystrobin" in x for x in p), pprint("BLOCK invented chemical + dose:", p)ok, p = audit_answer("Rotate fungicides [S9].", _m)assert not ok and any("does not exist" in x for x in p), pprint("BLOCK bad citation index")

## 15. Extractive fallback and the report entry point

In [ ]:
def extractive_report(cls, blocks, caveats):    lines = [f"# {pretty(cls).title()}", ""]    for key, label in BLOCK_LABELS.items():        lines.append(f"**{label}:**")        hits = blocks.get(key, [])        if not hits:            lines.append("The sources do not state this.")        else:            for h in hits:                lines.append(f"- {h['text']}  [{h['doc']} p.{h['page']}]")        lines.append("")    for c in caveats:        lines.append(f"> {c}")    return "\n".join(lines).strip()def generate_report(cls):    tier, blocks, caveats = retrieve_blocks(cls)    if cls in HEALTHY_CLASSES:        crop = cls.split("_")[0]        return {"class": cls, "tier": "healthy", "mode": "fixed", "audit_ok": True,                "text": f"No disease detected on this {crop} leaf. Continue routine "                        f"monitoring and normal cultural practices."}    if tier == "C" or not blocks:        return {"class": cls, "tier": tier, "mode": "none", "audit_ok": True,                "text": f"{pretty(cls).title()} is not covered by the knowledge base."}    if not LLM_OK:        return {"class": cls, "tier": tier, "mode": "extractive", "audit_ok": True,                "audit_problems": ["no LLM configured"],                "text": extractive_report(cls, blocks, caveats)}    user, meta = build_prompt(cls, blocks, caveats)    try:        answer = ask_llm(user)    except Exception as e:        return {"class": cls, "tier": tier, "mode": "extractive", "audit_ok": True,                "audit_problems": [f"api error: {type(e).__name__}: {str(e)[:120]}"],                "text": extractive_report(cls, blocks, caveats)}    ok, problems = audit_answer(answer, meta)    if ok:        body = answer + ("\n\n> " + " ".join(caveats) if caveats else "")        srcs = "\n".join(f"[S{i+1}] {h['doc']} p.{h['page']}" for i, h in enumerate(meta))        return {"class": cls, "tier": tier, "mode": "generated", "audit_ok": True,                "text": body + "\n\nSources:\n" + srcs}    return {"class": cls, "tier": tier, "mode": "extractive", "audit_ok": False,            "audit_problems": problems, "rejected_draft": answer,            "text": extractive_report(cls, blocks, caveats)}

## 16. Run it

In [ ]:
r = generate_report("tomato_early_blight")print(f"tier {r['tier']} | mode {r['mode']} | audit {r['audit_ok']} | model {LLM_MODEL}")if r.get("audit_problems"):    print("problems:", r["audit_problems"])print("-" * 78)print(r["text"])

In [ ]:
(OUT_DIR / "reports").mkdir(exist_ok=True)rows = []for cls in CLASSES:    rep = generate_report(cls)    (OUT_DIR / "reports" / f"{cls}.md").write_text(rep["text"], encoding="utf-8")    if not rep["audit_ok"]:        (OUT_DIR / "reports" / f"{cls}.REJECTED.txt").write_text(            rep.get("rejected_draft", ""), encoding="utf-8")    rows.append({"class": cls, "tier": rep["tier"], "mode": rep["mode"],                 "audit_ok": rep["audit_ok"], "chars": len(rep["text"]),                 "problems": "; ".join(rep.get("audit_problems", []))[:90]})    print(f"{cls:32s} {rep['tier']:8s} {rep['mode']:11s} audit={rep['audit_ok']}")df = pd.DataFrame(rows)df.to_csv(OUT_DIR / "rag_evaluation.csv", index=False)print("\n" + df["mode"].value_counts().to_string())failed = df[~df["audit_ok"]]if len(failed):    print(f"\n{len(failed)} audit failures - the rejected drafts are saved next to the "          f"reports:")    print(failed[["class", "problems"]].to_string(index=False))

## 17. Export`retriever.py` is emitted from an **explicit source template**, not from`inspect.getsource`. That matters: `getsource` reads IPython's linecache, which returnsstale or wrong source if a cell was edited and re-run, and fails outright under`nbconvert --execute`. The template can't do either.Drift between the template and the live notebook functions is caught instead by a**behavioural equivalence check** — the module is imported and its functions are comparedagainst the notebook's on sample inputs. If they disagree, this cell fails.**Download `plantcare_kb.db` too** — that single file *is* the vector database. No rebuildneeded to reuse it, but there is one catch worth knowing (verified, not guessed): areopened collection comes back in the **`released`** state, so a query fails with*"Collection is in state 'released'; call load() before search/get/query"* until you loadit. The fix is one line:```pythonfrom pymilvus import MilvusClientclient = MilvusClient(uri="plantcare_kb.db")client.load_collection("plantcare_kb")        # <-- required after every reopenclient.query("plantcare_kb", filter='disease == "tomato_target_spot"',             output_fields=["text", "doc"], limit=3)```

In [ ]:
manifest = {    "built_by": "PlantCare_NB02_Kaggle_MilvusLite",    "vocabulary": "v2", "section_types": SECTION_TYPES, "classes": CLASSES,    "healthy_classes": sorted(HEALTHY_CLASSES),    "n_chunks": len(chunks), "n_documents": len(parsed), "documents": doc_class,    "coverage": coverage,    "fallback_classes": [c for c, v in coverage.items() if v["verdict"] == "MISSING"],    "ocr_used": False, "sparse_in_milvus": SPARSE_IN_MILVUS,    "embed_model": ST_MODEL, "llm_provider": PROVIDER, "llm_model": LLM_MODEL,}HEADER = f'''"""Auto-generated by PlantCare_NB02 (Kaggle / Milvus Lite). Do not edit."""import re, json, math, time, requests, numpy as npfrom collections import Counter, defaultdictMILVUS_DB    = "plantcare_kb.db"COLLECTION   = {COLLECTION!r}ST_MODEL     = {ST_MODEL!r}QUERY_PREFIX = {QUERY_PREFIX!r}EMBED_DIM    = {EMBED_DIM}LLM_BASE_URL = {CFG["base_url"]!r}LLM_MODEL    = {LLM_MODEL!r}LLM_TEMP     = {LLM_TEMP}REQ_PAUSE    = {REQ_PAUSE}TOP_K_DENSE, TOP_K_SPARSE = {TOP_K_DENSE}, {TOP_K_SPARSE}RRF_K, FINAL_K, BLOCK_K   = {RRF_K}, {FINAL_K}, {BLOCK_K}BM25_K1, BM25_B = {BM25_K1}, {BM25_B}CONTEXT_BELOW   = {CONTEXT_BELOW}HEALTHY_CLASSES = {sorted(HEALTHY_CLASSES)!r}COVERAGE        = {coverage!r}SYSTEM          = {SYSTEM!r}SECTION_QUERIES = {SECTION_QUERIES!r}BLOCK_LABELS    = {BLOCK_LABELS!r}NUM_UNIT_RE = re.compile({NUM_UNIT_RE.pattern!r}, re.I)CHEM_RE     = re.compile({CHEM_RE.pattern!r}, re.I)TOKEN_RE    = re.compile(r"[a-z0-9]+")'''BODY = r'''def pretty(cls):    return cls.replace("_", " ")def tokenize(s):    return TOKEN_RE.findall(s.lower())def rrf(*rank_lists, k=RRF_K):    score = defaultdict(float)    for lst in rank_lists:        for rank, doc_id in enumerate(lst, start=1):            score[doc_id] += 1.0 / (k + rank)    return sorted(score.items(), key=lambda x: -x[1])def tier_for(cls):    if cls in HEALTHY_CLASSES:        return "healthy"    if COVERAGE.get(cls, {}).get("verdict") in ("OK", "THIN"):        return "A"    crop = cls.split("_")[0]    if any(c != cls and c not in HEALTHY_CLASSES and c.startswith(crop + "_")           and COVERAGE[c]["verdict"] in ("OK", "THIN") for c in COVERAGE):        return "B"    return "C"def build_prompt(cls, blocks, caveats):    src, meta = [], []    for name, hits in blocks.items():        for h in hits:            meta.append(h)            src.append("[S%d] (%s | %s p.%s) %s" % (len(meta), name, h["doc"],                                                    h["page"], h["text"]))    cav = ("\n\nIMPORTANT CONTEXT: " + " ".join(caveats)) if caveats else ""    headings = "\n".join("%s:" % l for l in BLOCK_LABELS.values())    user = ("Disease identified by the image classifier: %s\n\nSOURCES:\n%s%s\n\n"            "Write a short field advisory with exactly these headings:\n%s\n\n"            "One to three sentences per heading, in plain language a farmer can act on. "            "Cite after every sentence. If a heading has no supporting source, write "            "'The sources do not state this.'"            % (pretty(cls), "\n\n".join(src), cav, headings))    return user, metadef _norm(s):    return re.sub(r"[\s,]+", " ", s.lower()).strip()def audit_answer(answer, meta):    source_text = _norm(" ".join(h["text"] for h in meta))    compact = source_text.replace(" ", "")    problems = []    if not re.search(r"\[S\d+\]", answer):        problems.append("no citation markers at all")    for m in NUM_UNIT_RE.finditer(answer):        frag = _norm(m.group(0))        if frag not in source_text and frag.replace(" ", "") not in compact:            problems.append("unsupported quantity: %r" % m.group(0).strip())    for m in CHEM_RE.finditer(answer):        if m.group(0).lower() not in source_text:            problems.append("unsupported chemical: %r" % m.group(0))    for m in re.finditer(r"\[S(\d+)\]", answer):        if not (1 <= int(m.group(1)) <= len(meta)):            problems.append("citation [S%s] does not exist" % m.group(1))    return (not problems), problemsdef extractive_report(cls, blocks, caveats):    lines = ["# " + pretty(cls).title(), ""]    for key, label in BLOCK_LABELS.items():        lines.append("**%s:**" % label)        hits = blocks.get(key, [])        if not hits:            lines.append("The sources do not state this.")        else:            for h in hits:                lines.append("- %s  [%s p.%s]" % (h["text"], h["doc"], h["page"]))        lines.append("")    for c in caveats:        lines.append("> " + c)    return "\n".join(lines).strip()'''(OUT_DIR / "retriever.py").write_text(HEADER + BODY, encoding="utf-8")# ---- behavioural equivalence: the module must match the live notebook ----import importlib.utilspec = importlib.util.spec_from_file_location("retriever", OUT_DIR / "retriever.py")mod = importlib.util.module_from_spec(spec)spec.loader.exec_module(mod)assert mod.pretty("tomato_early_blight") == pretty("tomato_early_blight")assert mod.tokenize("Early Blight 2026!") == tokenize("Early Blight 2026!")assert mod.rrf([1, 2, 3], [3, 1]) == rrf([1, 2, 3], [3, 1])for _c in CLASSES:    assert mod.tier_for(_c) == tier_for(_c), _c_m = [{"text": "Mancozeb and Captan are protectants.", "doc": "t.pdf", "page": 1}]for probe in ["Use Mancozeb [S1].",              "Apply azoxystrobin at 2.5 g per litre [S1].",              "Rotate products [S9].",              "No citation here."]:    assert mod.audit_answer(probe, _m) == audit_answer(probe, _m), probe_t, _b, _cav = retrieve_blocks("tomato_early_blight")assert mod.extractive_report("tomato_early_blight", _b, _cav) == \       extractive_report("tomato_early_blight", _b, _cav)assert mod.build_prompt("tomato_early_blight", _b, _cav)[0] == \       build_prompt("tomato_early_blight", _b, _cav)[0]print("retriever.py matches the notebook on every checked function")# ---- the rest of the artefacts -----------------------------------------(OUT_DIR / "kb_chunks.json").write_text(    json.dumps({"manifest": manifest, "chunks": chunks}, ensure_ascii=False, indent=1),    encoding="utf-8")(OUT_DIR / "kb_manifest.json").write_text(json.dumps(manifest, indent=1), encoding="utf-8")np.save(OUT_DIR / "embeddings.npy", dense)pd.DataFrame(chunks).to_csv(OUT_DIR / "chunks.csv", index=False)print()for p in sorted(OUT_DIR.rglob("*")):    if p.is_file():        print(f"  {p.relative_to(OUT_DIR)}  ({p.stat().st_size/1024:.0f} KB)")print(f"\n  ../plantcare_kb.db  ({os.path.getsize(MILVUS_DB)/1024:.0f} KB)  <- the vector DB")

---## Reading the output honestly* `rag_evaluation.csv` → the `mode` column. `generated` means the LLM wrote it **and** the  audit passed. `extractive` means the API was unavailable or the audit caught something.* Any `audit_ok = False` row saves a `*.REJECTED.txt` next to the report. Read it — that  is the model inventing a number or a chemical, and the reason the audit exists.* Every disease class should be **tier A** with all five blocks filled.## What to download| File | For ||---|---|| `plantcare_kb.db` | the vector database itself — copy to your app, no rebuild || `kb_export/retriever.py` | import into notebook 03 / Flask || `kb_export/kb_chunks.json` | the chunks + manifest || `kb_export/reports/*.md` | the generated advisories |Milvus Lite is Linux-only. If your Flask app runs on Windows, use `embeddings.npy` +`chunks.csv` with the numpy path instead of the `.db` file.